## 0. Assumptions and inputs

This notebook deliberately starts with one basin (e.g. North Atlantic) and one simple Poisson specification.

Expected inputs:

1. `env_native`: monthly `xarray.Dataset` containing at least:
   - `pi` : thermodynamic potential intensity [m/s]
   - `rh600` : relative humidity at 600 hPa [% or fraction, but consistent]
   - `vws` : 850–200 hPa vertical wind shear magnitude [m/s]
   - `abs_vort850` : absolute vorticity magnitude [s-1]

2. `genesis`: a pandas DataFrame with one row per historical genesis event:
   - `storm_id`
   - `time`
   - `lat`
   - `lon`

The genesis definition must match the simulator target. If the simulator starts storms when they first reach tropical-storm intensity, use that same definition here.

3. `land_gdf`: a reasonably high-resolution land polygon GeoDataFrame (Natural Earth 10m / GSHHG / existing project coastline).

4. `basin_polygon`: the valid basin-water domain. This is important because `~land` alone can leave inland water bodies eligible.

In [1]:
from __future__ import annotations

import calendar
import os
import sys
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import geopandas as gpd
from shapely import box, contains_xy, union_all

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import poisson
import os
from datetime import date


# 1. Configuration

In [48]:
# Set this to the repository root. The notebook assumes the normal project layout:
# REPO_ROOT/
#   CODE/genesis_matrix.py
#   Land_ocean_masks/Land_ocean_mask_NA.txt
REPO_ROOT = os.path.dirname(os.getcwd())
sys.path.append(str(Path.cwd().parent))


import CODE.genesis_matrix as gm

# genesis_matrix.py currently derives __location__ from os.getcwd().
# Explicitly point it at the repository root so its existing mask loader is robust
# even if this notebook was launched from another directory.
gm.__location__ = str(REPO_ROOT)


ACTIVE_MONTHS = {
    "EP": (6, 7, 8, 9, 10, 11),
    "NA": (6, 7, 8, 9, 10, 11),
    "NI": (4, 5, 6, 9, 10, 11),
    "SI": (1, 2, 3, 4, 11, 12),
    "SP": (1, 2, 3, 4, 11, 12),
    "WP": (5, 6, 7, 8, 9, 10, 11),
}

NA_BOUDNARY = gm.BOUNDARIES_BASINS("NA")
@dataclass(frozen=True)
class GenesisConfig:
    basin: str = "NA"

    # Exposure scaling only changes the intercept and keeps coefficients readable.
    area_unit_km2: float = 100_000.0
    reference_month_days: float = 365.2425 / 12.0

    # Strict temporal holdout for the first experiment.
    first_test_year: int = 2016

    # Same native mask resolution already used by STORM.
    fine_res: float = 0.1

    active_months: tuple = (6, 7, 8, 9, 10, 11)
    lat_min : float = NA_BOUDNARY[0]
    lat_max : float = NA_BOUDNARY[1]
    lon_min : float = NA_BOUDNARY[2]
    lon_max : float = NA_BOUDNARY[3]


CFG = {basin: GenesisConfig(
    basin=basin,
    active_months=ACTIVE_MONTHS[basin],
    first_test_year=2016,
    lat_min=gm.BOUNDARIES_BASINS(basin)[0],
    lat_max=gm.BOUNDARIES_BASINS(basin)[1],
    lon_min=gm.BOUNDARIES_BASINS(basin)[2],
    lon_max=gm.BOUNDARIES_BASINS(basin)[3],
) for basin in ["NA", "WP", "EP", "SP", "NI", "SI"]}


ROWS = [int(abs(cfg_basin.lat_max - cfg_basin.lat_min)) for cfg_basin in CFG.values()]
COLS = [int(abs(cfg_basin.lon_max - cfg_basin.lon_min)) for cfg_basin in CFG.values()]

LAT = {cfg_basin.basin: gm.basin_cell_centers(cfg_basin.basin, ROWS[i], COLS[i])[0][ :, 0] for i, cfg_basin in enumerate(CFG.values())}
LON = {cfg_basin.basin: gm.basin_cell_centers(cfg_basin.basin, ROWS[i], COLS[i])[1][0, :] for i, cfg_basin in enumerate(CFG.values())}
  # north -> south, matching genesis_matrix.py
  # west -> east, 0..360 convention

print(CFG)
print([f"{cfg.basin}: {rows} x {cols} 1-degree cells" for cfg, rows, cols in zip(CFG.values(), ROWS, COLS)])


{'NA': GenesisConfig(basin='NA', area_unit_km2=100000.0, reference_month_days=30.436875, first_test_year=2016, fine_res=0.1, active_months=(6, 7, 8, 9, 10, 11), lat_min=5, lat_max=60, lon_min=255, lon_max=360), 'WP': GenesisConfig(basin='WP', area_unit_km2=100000.0, reference_month_days=30.436875, first_test_year=2016, fine_res=0.1, active_months=(5, 6, 7, 8, 9, 10, 11), lat_min=5, lat_max=60, lon_min=100, lon_max=180), 'EP': GenesisConfig(basin='EP', area_unit_km2=100000.0, reference_month_days=30.436875, first_test_year=2016, fine_res=0.1, active_months=(6, 7, 8, 9, 10, 11), lat_min=5, lat_max=60, lon_min=180, lon_max=285), 'SP': GenesisConfig(basin='SP', area_unit_km2=100000.0, reference_month_days=30.436875, first_test_year=2016, fine_res=0.1, active_months=(1, 2, 3, 4, 11, 12), lat_min=-60, lat_max=-5, lon_min=135, lon_max=240), 'NI': GenesisConfig(basin='NI', area_unit_km2=100000.0, reference_month_days=30.436875, first_test_year=2016, fine_res=0.1, active_months=(4, 5, 6, 9, 10,

## NaturalEarth mask

In [ ]:
import pandas as pd
import geopandas
import matplotlib.pyplot as plt
from geodatasets import get_path


land_gdf = geopandas.read_file(get_path("naturalearth.land"))


## Load IBTRACS STORMS

In [33]:
REPO_ROOT = os.path.dirname(os.getcwd())

year = np.load(os.path.join(REPO_ROOT, "YEARLIST_INTERP.npy"), allow_pickle=True).item()
lat = np.load(os.path.join(REPO_ROOT, "LATLIST_INTERP.npy"), allow_pickle=True).item()
lon = np.load(os.path.join(REPO_ROOT, "LONLIST_INTERP.npy"), allow_pickle=True).item()
pressure = np.load(
    os.path.join(REPO_ROOT, "PRESLIST_INTERP.npy"), allow_pickle=True
).item()
month = np.load(
    os.path.join(REPO_ROOT, "MONTHLIST_INTERP.npy"), allow_pickle=True
).item()
speed = np.load(
    os.path.join(REPO_ROOT, "WINDLIST_INTERP.npy"), allow_pickle=True
).item()

idx_clean = [idx for idx, i in enumerate(lat) if len(lat[i]) > 0]
year = [year[i] for i in idx_clean]
month = [month[i] for i in idx_clean]
lat = [lat[i] for i in idx_clean]
lon = [lon[i] for i in idx_clean]
pressure = [pressure[i] for i in idx_clean]
speed = [speed[i] for i in idx_clean]


genesis = pd.DataFrame(
    {
        "stormid": idx_clean,
        "time": [
            date(int(year[i][0]), int(month[i][0]), 1) for i in range(len(idx_clean))
        ],
        "latitude": [lat_i[0] for lat_i in lat],
        "longitude": [lon_i[0] for lon_i in lon],
        "speed": [speed_i[0] for speed_i in speed],
    }
)



genesis_basin = {basin: genesis.loc[
    (genesis["latitude"] > cfg_basin.lat_min)
    & (genesis["longitude"] > cfg_basin.lon_min)
    & (genesis["latitude"] < cfg_basin.lat_max)
    & (genesis["longitude"] < cfg_basin.lon_max),
] for basin, cfg_basin in CFG.items()
}

# Basin Specific Preparation

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


path_files = Path(
    "/home/mbaldacchino/data/SIENA-IH-STORM_1980_2025_VF/SIENA-IH-STORM/env_yearly"
)
# Files we actually need for the first genesis model.
NAME_MAP = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "VORT850": "vort850",  # relative vorticity [s-1]
    "SST": "sst",
}

NAME_MODEL = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "ETA850": "eta850",  # absolute vorticity capped [s-1]
    "W500": "w500",  # vertical velocity at 500hPa
}


START_YEAR = 1980
END_YEAR = 2025


def parse_env_filename(path):
    """
    Parse:
        VWS_2000_6.npy
        VMAX_PI_2000_6.npy
        VORT850_2018_10.npy

    Returns:
        variable, year, month
    """
    name, year, month = path.stem.rsplit("_", 2)

    return name, int(year), int(month)

In [ ]:
def load_environment_dataset(
    path_files,
    name_map,
    start_year=1980,
    end_year=2024,
):
    path_files = Path(path_files)

    # Your arrays are the global 0.25° fields used elsewhere in the repo:
    # 721 latitudes: 90 -> -90
    # 1440 longitudes: 0 -> 359.75
    lat = np.linspace(90.0, -90.0, 721)
    lon = np.arange(1440) * 0.25

    variables = {}

    for file_name, xr_name in name_map.items():
        records = []

        for file in path_files.glob(f"{file_name}_*.npy"):
            name, year, month = parse_env_filename(file)

            if name != file_name:
                continue

            if not (start_year <= year <= end_year):
                continue

            records.append((year, month, file))

        records.sort(key=lambda x: (x[0], x[1]))

        if not records:
            raise FileNotFoundError(f"No files found for {file_name}")

        arrays = []
        times = []

        for year, month, file in records:
            arr = np.load(file)

            if arr.shape != (721, 1440):
                raise ValueError(f"{file.name}: expected (721, 1440), got {arr.shape}")

            arrays.append(arr)

            times.append(pd.Timestamp(year=year, month=month, day=1))

        variables[xr_name] = xr.DataArray(
            np.stack(arrays, axis=0),
            dims=("valid_time", "latitude", "longitude"),
            coords={
                "valid_time": pd.DatetimeIndex(times),
                "latitude": lat,
                "longitude": lon,
            },
            name=xr_name,
        )

    return xr.Dataset(variables)


env_native = load_environment_dataset(
    path_files=path_files,
    name_map=NAME_MAP,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

OMEGA = 7.2921e-5

f = 2 * OMEGA * np.sin(np.deg2rad(env_native.latitude))

eta = env_native["vort850"] + f

env_native["eta850"] = xr.where(
    eta > 0,
    eta,
    0.0,
).clip(max=5e-5)

# TEMP FIX ADD W500
vv = xr.open_dataset("/home/mbaldacchino/data/vertical_velocity.grib")
vv = vv.rename_vars({"w": "w500"})

env_native = env_native.merge(
    vv.swap_dims({"time": "valid_time"})
    .sel(valid_time=slice("1980-01-01", "2025-12-01"))
    .drop_vars(["number", "time", "step"])
)

del vv


### NA Specific

In [ ]:
# COMPUTE rSST

JJASON = [6, 7, 8, 9, 10, 11]
ASO = [8, 9, 10]


def area_weighted_mean(da):
    weights = np.cos(np.deg2rad(da.latitude))

    return da.weighted(weights).mean(["latitude", "longitude"])


def area_weighted_mean_quantile(
    da,
    quantile=0.7,
):
    """
    Area-weighted mean of values above an area-weighted quantile.

    For SSTW30:
        quantile=0.7
    returns approximately the area-weighted mean SST of the
    warmest 30% of the domain.

    Assumes:
    - regular lat/lon grid
    - land points are NaN
    """

    weights = np.cos(np.deg2rad(da.latitude))

    # Area-weighted quantile at each time step
    threshold = da.weighted(weights).quantile(
        quantile,
        dim=["latitude", "longitude"],
        skipna=True,
    )

    # Keep only cells warmer than the weighted threshold
    warm = da.where(da >= threshold)

    # Area-weighted mean of those warmest cells
    return warm.weighted(weights).mean(
        ["latitude", "longitude"],
        skipna=True,
    )


MDR_SST = area_weighted_mean(
    env_native.sst.sel(
        latitude=slice(
            MDR_REGION["lat_max"],
            MDR_REGION["lat_min"],
        ),
        longitude=slice(
            MDR_REGION["lon_min"],
            MDR_REGION["lon_max"],
        ),
    )
)

TROPICS_SST = area_weighted_mean(
    env_native.sst.sel(
        latitude=slice(
            TROPICS_REGION["lat_max"],
            TROPICS_REGION["lat_min"],
        ),
        longitude=slice(
            TROPICS_REGION["lon_min"],
            TROPICS_REGION["lon_max"],
        ),
    )
)

SST_W30 = area_weighted_mean_quantile(
    env_native.sst.sel(
        latitude=slice(
            TROPICS_REGION["lat_max"],
            TROPICS_REGION["lat_min"],
        ),
        longitude=slice(
            TROPICS_REGION["lon_min"],
            TROPICS_REGION["lon_max"],
        ),
    ),
    quantile=0.70,
)

ds_rsst = xr.Dataset(
    {
        "mdr_sst": MDR_SST,
        "tropics_sst": TROPICS_SST,
        "rsst": MDR_SST - TROPICS_SST,
        "rsst_bis": MDR_SST - SST_W30,
    }
)

ds_jjason = ds_rsst.sel(valid_time=ds_rsst.valid_time.dt.month.isin(JJASON))
weights = ds_jjason.valid_time.dt.days_in_month
yearly = (ds_jjason * weights).groupby("valid_time.year").sum(
    "valid_time"
) / weights.groupby("valid_time.year").sum("valid_time")


ds_aso = ds_rsst.sel(valid_time=ds_rsst.valid_time.dt.month.isin(ASO))
weights = ds_aso.valid_time.dt.days_in_month
yearly_aso = (ds_aso * weights).groupby("valid_time.year").sum(
    "valid_time"
) / weights.groupby("valid_time.year").sum("valid_time")

yearly_df = yearly.to_dataframe().reset_index()
yearly_aso = yearly_aso.to_dataframe().reset_index()


env_native = env_native.drop_vars("sst")


# Grid Definition and exact spherical cell areas

In [40]:
EARTH_RADIUS_KM = 6371.0088


def spherical_1deg_area_km2(lat_center):
    """Exact spherical area of a 1° x 1° lat/lon cell centered at lat_center."""
    lat_lo = np.deg2rad(np.asarray(lat_center) - 0.5)
    lat_hi = np.deg2rad(np.asarray(lat_center) + 0.5)
    dlon = np.deg2rad(1.0)
    return EARTH_RADIUS_KM**2 * dlon * (np.sin(lat_hi) - np.sin(lat_lo))


def build_existing_storm_exposure(
    basin: str,
    repo_root: Path,
) -> xr.Dataset:
    lat0, lat1, lon0, lon1 = gm.BOUNDARIES_BASINS(basin)
    rows = int(abs(lat1 - lat0))
    cols = int(abs(lon1 - lon0))

    # Existing authoritative 1° genesis-validity mask.
    ocean_valid = gm._genesis_ocean_mask_1deg(basin, (rows, cols))

    # Recover fractional ocean coverage from the exact same 0.1° mask.
    mask_path = os.path.join(repo_root, f"Land_ocean_masks/Land_ocean_mask_{basin}.txt")
    land10 = np.loadtxt(mask_path)

    # Same endpoint handling as _genesis_ocean_mask_1deg.
    land10 = land10[: rows * 10, : cols * 10]
    land_fraction = land10.reshape(rows, 10, cols, 10).mean(axis=(1, 3))
    ocean_fraction_raw = 1.0 - land_fraction

    # Preserve the exact existing genesis admissibility rule, including NA/EP split.
    eligible_fraction = np.where(ocean_valid, ocean_fraction_raw, 0.0)

    LAT2D, LON2D = gm.basin_cell_centers(basin, rows, cols)
    lat = LAT2D[:, 0]
    lon = LON2D[0, :]

    area_lat = spherical_1deg_area_km2(lat)
    cell_area = np.broadcast_to(area_lat[:, None], (rows, cols)).copy()
    eligible_area = cell_area * eligible_fraction

    return xr.Dataset(
        {
            "ocean_valid": (("latitude", "longitude"), ocean_valid),
            "raw_ocean_fraction": (("latitude", "longitude"), ocean_fraction_raw),
            "eligible_fraction": (("latitude", "longitude"), eligible_fraction),
            "cell_area_km2": (("latitude", "longitude"), cell_area),
            "eligible_area_km2": (("latitude", "longitude"), eligible_area),
        },
        coords={"latitude": lat, "longitude": lon},
        attrs={"basin": basin, "rows": rows, "cols": cols}
    )


coarse_exposure = [build_existing_storm_exposure(basin, REPO_ROOT) for basin in CFG]


Quickly visualize

In [ ]:
for coarse_exposure_i in coarse_exposure:
    coarse_exposure_i.eligible_area_km2.plot()
    plt.title(f"Eligible Area for Basin {coarse_exposure_i.attrs.get('basin', 'Unknown')}")
    plt.show()


## 4. Put environmental fields on the exact existing STORM 1° grid

Keep the project's 0–360° longitude convention.

The target coordinates come directly from `gm.basin_cell_centers()`, so the
environmental arrays, observed genesis counts, existing masks, and simulated
genesis probabilities all use the same row/column convention.

In [ ]:
def normalize_lon_360(ds: xr.Dataset, lon_name: str = "longitude") -> xr.Dataset:
    lon_new = ds[lon_name] % 360
    return ds.assign_coords({lon_name: lon_new}).sortby(lon_name)


def standardize_month_time(ds: xr.Dataset) -> xr.Dataset:
    t = pd.DatetimeIndex(ds.valid_time.values).to_period("M").to_timestamp()
    return ds.assign_coords(valid_time=t)


from time import time


def to_existing_storm_grid(ds: xr.Dataset, cfg: GenesisConfig) -> xr.Dataset:
    a = time()
    ds = normalize_lon_360(ds)
    print(time() - a)
    ds = standardize_month_time(ds)
    print(time() - a)
    ds = ds.sel(valid_time=ds.valid_time.dt.month.isin(ACTIVE_MONTHS[cfg.basin]))

    # xarray interpolation is safest with monotonic source coordinates.
    ds = ds.sortby("latitude").sortby("longitude")
    print(time() - a)
    # Target latitude is intentionally north->south to match genesis_matrix.py.
    out = ds.interp(latitude=LAT, longitude=LON)
    print(time() - a)
    return out


# Example if the fields are already suitable for direct interpolation:
env_1deg = to_existing_storm_grid(env_native, CFG)[
    list(NAME_MODEL.values())
]  # [VARS_NAME]
#
# If your 0.25° preprocessing already has a preferred area/ocean-weighted
# 0.25° -> 1° aggregation, use that instead, but end on exactly LAT/LON above.


## 5. Historical genesis events

`genesis` should already contain exactly one row per storm under the definition you want the synthetic model to reproduce.

Recommended for compatibility with a STORM-like simulator:

- keep tropical/subtropical nature according to a fixed documented rule;
- identify the first fix at which the storm reaches the simulator's minimum wind threshold;
- use a single wind convention consistently;
- assign that time/location as the genesis event.

Do **not** fit a model to disturbance genesis if your simulator starts at tropical-storm genesis.

In [ ]:
def normalize_lon_values(lon_values):
    """Use the same 0..360 convention as the STORM basin definitions."""
    return np.asarray(lon_values, dtype=float) % 360


def prepare_genesis_events(genesis: pd.DataFrame, cfg: GenesisConfig) -> pd.DataFrame:
    lat0, lat1, lon0, lon1 = gm.BOUNDARIES_BASINS(cfg.basin)

    g = genesis.copy()
    g["time"] = pd.to_datetime(g["time"])
    g["time_month"] = g["time"].dt.to_period("M").dt.to_timestamp()
    g["year"] = g["time"].dt.year
    g["month"] = g["time"].dt.month
    g["longitude"] = normalize_lon_values(g["longitude"])

    g = g[
        g["month"].isin(ACTIVE_MONTHS[cfg.basin])
        & g["latitude"].ge(lat0)
        & g["latitude"].lt(lat1)
        & g["longitude"].ge(lon0)
        & g["longitude"].lt(lon1)
    ].copy()

    return g


genesis = [prepare_genesis_events(genesis_basin[basin_cfg], basin_cfg) for basin_cfg in CFG.values()]

## 6. Assign observed genesis events to the existing 1° STORM cells

Rows are north-to-south and columns west-to-east, exactly as in
`basin_cell_centers()` and `_genesis_ocean_mask_1deg()`.

In [ ]:
def assign_cells(df: pd.DataFrame, basin: GenesisConfig) -> pd.DataFrame:
    lat0, lat1, lon0, lon1 = basin.lat_min, basin.lat_max, basin.lon_min, basin.lon_max
    rows = int(abs(lat1 - lat0))
    cols = int(abs(lon1 - lon0))

    out = df.copy()

    # Row 0 is [lat1-1, lat1), centered at lat1-0.5.
    ilat = np.floor(lat1 - out["latitude"].to_numpy()).astype(int)
    ilon = np.floor(out["longitude"].to_numpy() - lon0).astype(int)

    valid = (ilat >= 0) & (ilat < rows) & (ilon >= 0) & (ilon < cols)

    out = out.loc[valid].copy()
    out["ilat"] = ilat[valid]
    out["ilon"] = ilon[valid]
    out["cell_lat"] = LAT[basin.basin][out["ilat"].to_numpy()]
    out["cell_lon"] = LON[basin.basin][out["ilon"].to_numpy()]

    return out


genesis = [assign_cells(genesis[i], basin_cfg) for i, basin_cfg in enumerate(CFG.values())]


### Bucket regions

In [ ]:
BUCKET_REGIONS = {
    "NA": {
        "gulf": (18, 31, 260, 282),
        "south_caribbean": (7, 16, 275, 300),
        "north_caribbean": (16, 25, 275, 300),
        "mdr": (None, 20, 300, 350),
        "subtropical": (20, None, 300, 360),
    },
    "WP": {
        "scs": (7, 30, 100,  120),
        "nwp_nw": (17, 40, 120, 140),
        "nwp_sw": (7, 17, 120, 140),
        "nwp_se": (7, 17, 140, 160),
        "nwp_ne": (17, 40, 140, 160),
    },
    "EP" : {
        "mdr" : (5, 20, 220, 280),
        "subtropical" : (None, None, None, 280),
    },
    "NI" : {
        "arabian_sea" : (5, 28, 47,77),
        "bay_of_bengal" : (5, None, 77, None),
    },
    "SI" : {
        "southwest_indian_ocean" : (-35, -5, 25, 60),
        "central_south_indian_ocean" : (-35, -5, 60, 90),
        "northwest_autralia" : (-35, -5, 90, None),
    },
    "SP" : {
        "western_south_pacific" : (-31, -5, 150, 170),
        'central_south_pacific' : (-31, -5, 170, 190),
        'eastern_south_pacific' : (-31, -5, 190, 240)
    }

}


def assign_genesis_region(lat, lon, basin="NA"):
    """
    lon in 0..360 convention.
    """

    if basin in BUCKET_REGIONS:
        for region, (lat_min, lat_max, lon_min, lon_max) in BUCKET_REGIONS[basin].items():
            if (
                (lat_min is None or lat >= lat_min)
                and (lat_max is None or lat < lat_max)
                and (lon_min is None or lon >= lon_min)
                and (lon_max is None or lon < lon_max)
            ):
                return region
    else:
        raise ValueError(f"Unknown basin {basin}")


genesis["region"] = [
    assign_genesis_region(lat, lon, basin)
    for lat, lon, basin in zip(
        genesis["latitude"],
        genesis["longitude"],
        genesis["basin"],
    )
    )
]




for i,genesis_i in enumerate(genesis):
    genesis_i["region"] = [
        assign_genesis_region(lat, lon, CFG[basin].basin)
        for lat, lon, basin in zip(
            genesis_i["latitude"],
            genesis_i["longitude"],
        )
    ]




    print(f"Basin {genesis_i['basin'].iloc[0]}: {len(genesis_i)} genesis events")

## 7. Build the cell-month modelling table

Every eligible 1° cell and every historical active-season month is one row.

The response is:

`genesis_count`

Most rows will be zero. That is expected and is **not** by itself evidence of zero inflation.

Exposure is:

\[
E_{iym}
=

rac{A^{eligible}_i}{100000\ km^2}
	imes

rac{days(m)}{30.44}.
\]

The scale constants only make the intercept readable.

In [ ]:
def build_model_table(
    env_1deg: xr.Dataset,
    coarse_exposure: xr.Dataset,
    genesis_cells: pd.DataFrame,
    cfg: GenesisConfig,
) -> pd.DataFrame:
    missing = [v for v in list(NAME_MODEL.values()) if v not in env_1deg]
    if missing:
        raise KeyError(f"Missing environmental variables: {missing}")

    env_df = env_1deg[list(NAME_MODEL.values())].to_dataframe().reset_index()
    env_df["time"] = pd.to_datetime(env_df["valid_time"])
    env_df["year"] = env_df["time"].dt.year
    env_df["month"] = env_df["time"].dt.month

    exposure_df = (
        coarse_exposure[["ocean_valid", "eligible_area_km2", "eligible_fraction"]]
        .to_dataframe()
        .reset_index()
    )

    df = env_df.merge(exposure_df, on=["latitude", "longitude"], how="left")

    counts = (
        genesis_cells.groupby(["time_month", "cell_lat", "cell_lon"])
        .size()
        .rename("genesis_count")
        .reset_index()
        .rename(
            columns={
                "time_month": "time",
                "cell_lat": "latitude",
                "cell_lon": "longitude",
            }
        )
    )

    df = df.merge(counts, on=["time", "latitude", "longitude"], how="left")
    df["genesis_count"] = df["genesis_count"].fillna(0).astype(int)

    days = df["time"].dt.days_in_month.astype(float)
    df["exposure"] = (df["eligible_area_km2"] / cfg.area_unit_km2) * (
        days / cfg.reference_month_days
    )

    valid = (
        df["ocean_valid"].fillna(False)
        & (df["exposure"] > 0)
        & np.isfinite(df[list(NAME_MODEL.values())]).all(axis=1)
    )

    df = df.loc[valid].copy()
    df["log_exposure"] = np.log(df["exposure"])

    return df


model_df = build_model_table(env_1deg, coarse_exposure, genesis, CFG)
model_df.head()


# 10. Standardize data

In [ ]:
def fit_standardizer(train: pd.DataFrame, features: list[str]):
    params = {}
    for col in features:
        mu = train[col].mean()
        sd = train[col].std()
        if not np.isfinite(sd) or sd == 0:
            raise ValueError(f"Invalid standard deviation for {col}: {sd}")
        params[col] = (mu, sd)
    return params


def apply_standardizer(df: pd.DataFrame, params):
    out = df.copy()
    for col, (mu, sd) in params.items():
        out[f"z_{col}"] = (out[col] - mu) / sd
    return out


scaler = fit_standardizer(train, list(NAME_MODEL.values()))
train = apply_standardizer(train, scaler)
test = apply_standardizer(test, scaler)


# 11. Fit Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson
from patsy import bs


## Helpers

In [ ]:
def prepare_clipped_eta(
    train,
    threshold,
):
    train_tmp = train.copy()

    train_tmp["eta_clip"] = np.minimum(
        train_tmp["eta850"],
        threshold,
    )


    # Standardization fitted ONLY on training period
    mean_eta = train_tmp["eta_clip"].mean()
    std_eta = train_tmp["eta_clip"].std()

    train_tmp["z_eta_clip"] = (train_tmp["eta_clip"] - mean_eta) / std_eta

    return train_tmp


## A -  NA

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


path_files = Path(
    "/home/mbaldacchino/data/SIENA-IH-STORM_1980_2025_VF/SIENA-IH-STORM/env_yearly"
)

# Files we actually need for the first genesis model.
NAME_MAP = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "VORT850": "vort850",  # relative vorticity [s-1]
    "SST": "sst",
}

NAME_MODEL = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "ETA850": "eta850",  # absolute vorticity capped [s-1]
    "W500": "w500",  # vertical velocity at 500hPa
}

START_YEAR = 1980
END_YEAR = 2025


def parse_env_filename(path):
    """
    Parse:
        VWS_2000_6.npy
        VMAX_PI_2000_6.npy
        VORT850_2018_10.npy

    Returns:
        variable, year, month
    """
    name, year, month = path.stem.rsplit("_", 2)

    return name, int(year), int(month)


In [ ]:
def load_environment_dataset(
    path_files,
    name_map,
    start_year=1980,
    end_year=2024,
):
    path_files = Path(path_files)

    # Your arrays are the global 0.25° fields used elsewhere in the repo:
    # 721 latitudes: 90 -> -90
    # 1440 longitudes: 0 -> 359.75
    lat = np.linspace(90.0, -90.0, 721)
    lon = np.arange(1440) * 0.25

    variables = {}

    for file_name, xr_name in name_map.items():
        records = []

        for file in path_files.glob(f"{file_name}_*.npy"):
            name, year, month = parse_env_filename(file)

            if name != file_name:
                continue

            if not (start_year <= year <= end_year):
                continue

            records.append((year, month, file))

        records.sort(key=lambda x: (x[0], x[1]))

        if not records:
            raise FileNotFoundError(f"No files found for {file_name}")

        arrays = []
        times = []

        for year, month, file in records:
            arr = np.load(file)

            if arr.shape != (721, 1440):
                raise ValueError(f"{file.name}: expected (721, 1440), got {arr.shape}")

            arrays.append(arr)

            times.append(pd.Timestamp(year=year, month=month, day=1))

        variables[xr_name] = xr.DataArray(
            np.stack(arrays, axis=0),
            dims=("valid_time", "latitude", "longitude"),
            coords={
                "valid_time": pd.DatetimeIndex(times),
                "latitude": lat,
                "longitude": lon,
            },
            name=xr_name,
        )

    return xr.Dataset(variables)


env_native = load_environment_dataset(
    path_files=path_files,
    name_map=NAME_MAP,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

OMEGA = 7.2921e-5

f = 2 * OMEGA * np.sin(np.deg2rad(env_native.latitude))

eta = env_native["vort850"] + f

env_native["eta850"] = xr.where(
    eta > 0,
    eta,
    0.0,
).clip(max=5e-5)

# TEMP FIX ADD W500
vv = xr.open_dataset("/home/mbaldacchino/data/vertical_velocity.grib")
vv = vv.rename_vars({"w": "w500"})

env_native = env_native.merge(
    vv.swap_dims({"time": "valid_time"})
    .sel(valid_time=slice("1980-01-01", "2025-12-01"))
    .drop_vars(["number", "time", "step"])
)

del vv


In [ ]:
# COMPUTE rSST

JJASON = [6, 7, 8, 9, 10, 11]
ASO = [8, 9, 10]


def area_weighted_mean(da):
    weights = np.cos(np.deg2rad(da.latitude))

    return da.weighted(weights).mean(["latitude", "longitude"])


def area_weighted_mean_quantile(
    da,
    quantile=0.7,
):
    """
    Area-weighted mean of values above an area-weighted quantile.

    For SSTW30:
        quantile=0.7
    returns approximately the area-weighted mean SST of the
    warmest 30% of the domain.

    Assumes:
    - regular lat/lon grid
    - land points are NaN
    """

    weights = np.cos(np.deg2rad(da.latitude))

    # Area-weighted quantile at each time step
    threshold = da.weighted(weights).quantile(
        quantile,
        dim=["latitude", "longitude"],
        skipna=True,
    )

    # Keep only cells warmer than the weighted threshold
    warm = da.where(da >= threshold)

    # Area-weighted mean of those warmest cells
    return warm.weighted(weights).mean(
        ["latitude", "longitude"],
        skipna=True,
    )


MDR_SST = area_weighted_mean(
    env_native.sst.sel(
        latitude=slice(
            MDR_REGION["lat_max"],
            MDR_REGION["lat_min"],
        ),
        longitude=slice(
            MDR_REGION["lon_min"],
            MDR_REGION["lon_max"],
        ),
    )
)

TROPICS_SST = area_weighted_mean(
    env_native.sst.sel(
        latitude=slice(
            TROPICS_REGION["lat_max"],
            TROPICS_REGION["lat_min"],
        ),
        longitude=slice(
            TROPICS_REGION["lon_min"],
            TROPICS_REGION["lon_max"],
        ),
    )
)

SST_W30 = area_weighted_mean_quantile(
    env_native.sst.sel(
        latitude=slice(
            TROPICS_REGION["lat_max"],
            TROPICS_REGION["lat_min"],
        ),
        longitude=slice(
            TROPICS_REGION["lon_min"],
            TROPICS_REGION["lon_max"],
        ),
    ),
    quantile=0.70,
)

ds_rsst = xr.Dataset(
    {
        "mdr_sst": MDR_SST,
        "tropics_sst": TROPICS_SST,
        "rsst": MDR_SST - TROPICS_SST,
        "rsst_bis": MDR_SST - SST_W30,
    }
)

ds_jjason = ds_rsst.sel(valid_time=ds_rsst.valid_time.dt.month.isin(JJASON))
weights = ds_jjason.valid_time.dt.days_in_month
yearly = (ds_jjason * weights).groupby("valid_time.year").sum(
    "valid_time"
) / weights.groupby("valid_time.year").sum("valid_time")


ds_aso = ds_rsst.sel(valid_time=ds_rsst.valid_time.dt.month.isin(ASO))
weights = ds_aso.valid_time.dt.days_in_month
yearly_aso = (ds_aso * weights).groupby("valid_time.year").sum(
    "valid_time"
) / weights.groupby("valid_time.year").sum("valid_time")

yearly_df = yearly.to_dataframe().reset_index()
yearly_aso = yearly_aso.to_dataframe().reset_index()


env_native = env_native.drop_vars("sst")


In [ ]:
train_eta = prepare_clipped_eta(
    train,
    3.5e-5,
)

In [ ]:
region_grid = (
    train_eta[["latitude", "longitude", "region"]]
    .drop_duplicates(["latitude", "longitude"])
    .pivot(
        index="latitude",
        columns="longitude",
        values="region",
    )
)

import numpy as np
from scipy.ndimage import gaussian_filter

regions = sorted(train_eta["region"].unique())

hard_masks = {}

for region in regions:
    hard_masks[region] = (region_grid.values == region).astype(float)


SIGMA_REGION = 3.0


In [ ]:

soft_masks = {}

for region in regions:
    soft_masks[region] = gaussian_filter(
        hard_masks[region],
        sigma=3.0,
        mode="nearest",
    )

total = np.zeros_like(next(iter(soft_masks.values())))

for field in soft_masks.values():
    total += field

total = np.where(
    total > 0,
    total,
    np.nan,
)

for region in regions:
    soft_masks[region] = soft_masks[region] / total


weights_df = []

lat_values = region_grid.index.to_numpy()
lon_values = region_grid.columns.to_numpy()

for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        row = {
            "latitude": lat,
            "longitude": lon,
        }

        for region in regions:
            row[f"w_{region}"] = soft_masks[region][i, j]

        weights_df.append(row)

weights_df = pd.DataFrame(weights_df)

train_soft = train_eta.merge(
    weights_df,
    on=["latitude", "longitude"],
    how="left",
)


In [ ]:
train_soft["rsst"] = train_soft["year"].map(yearly_df.set_index("year")["rsst"])
train_soft["rsst"] = (train_soft["rsst"] - np.mean(train_soft["rsst"])) / np.std(
    train_soft["rsst"]
)

REFERENCE_REGION = "subtropical"

varying_regions = [r for r in regions if r != REFERENCE_REGION]
region_terms = " + ".join(f"w_{r}" for r in varying_regions)


FORMULA_TREND = (
    "genesis_count ~ C(month)"
    " + z_pi"
    " + z_rh600"
    " + z_vws"
    " + z_eta_clip"
    " + z_w500"
    f" + {region_terms}"
    " + rsst"
)

m_trend = smf.glm(
    formula=FORMULA_TREND,
    data=train_soft,
    family=sm.families.Poisson(),
    offset=train_soft["log_exposure"],
).fit()


## B - WP

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


path_files = Path(
    "/home/mbaldacchino/data/SIENA-IH-STORM_1980_2025_VF/SIENA-IH-STORM/env_yearly"
)

# Files we actually need for the first genesis model.
NAME_MAP = {
    "VMAX_PI": "pi",
    "VWS": "vws",
    "RH600": "rh600",
    "VORT850": "vort850",
    "SST": "sst",
    # Adapt the left-hand names to your actual .npy filenames:
    # "T2M": "t2m",
    "MSLP": "mslp",
    # "U850": "u850",
}

NAME_MODEL = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "ETA850": "eta850",  # absolute vorticity capped [s-1]
    "W500": "w500",  # vertical velocity at 500hPa
}

START_YEAR = 1980
END_YEAR = 2024


def parse_env_filename(path):
    """
    Parse:
        VWS_2000_6.npy
        VMAX_PI_2000_6.npy
        VORT850_2018_10.npy

    Returns:
        variable, year, month
    """
    name, year, month = path.stem.rsplit("_", 2)

    return name, int(year), int(month)


def load_environment_dataset(
    path_files,
    name_map,
    start_year=1980,
    end_year=2024,
):
    path_files = Path(path_files)

    # Your arrays are the global 0.25° fields used elsewhere in the repo:
    # 721 latitudes: 90 -> -90
    # 1440 longitudes: 0 -> 359.75
    lat = np.linspace(90.0, -90.0, 721)
    lon = np.arange(1440) * 0.25

    variables = {}

    for file_name, xr_name in name_map.items():
        records = []

        for file in path_files.glob(f"{file_name}_*.npy"):
            name, year, month = parse_env_filename(file)

            if name != file_name:
                continue

            if not (start_year <= year <= end_year):
                continue

            records.append((year, month, file))

        records.sort(key=lambda x: (x[0], x[1]))

        if not records:
            raise FileNotFoundError(f"No files found for {file_name}")

        arrays = []
        times = []

        for year, month, file in records:
            arr = np.load(file)

            if arr.shape != (721, 1440):
                raise ValueError(f"{file.name}: expected (721, 1440), got {arr.shape}")

            arrays.append(arr)

            times.append(pd.Timestamp(year=year, month=month, day=1))

        variables[xr_name] = xr.DataArray(
            np.stack(arrays, axis=0),
            dims=("valid_time", "latitude", "longitude"),
            coords={
                "valid_time": pd.DatetimeIndex(times),
                "latitude": lat,
                "longitude": lon,
            },
            name=xr_name,
        )

    return xr.Dataset(variables)


env_native = load_environment_dataset(
    path_files=path_files,
    name_map=NAME_MAP,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

OMEGA = 7.2921e-5

f = 2 * OMEGA * np.sin(np.deg2rad(env_native.latitude))

eta = env_native["vort850"] + f

env_native["eta850"] = xr.where(
    eta > 0,
    eta,
    0.0,
).clip(max=5e-5)

# TEMP FIX ADD W500
vv = xr.open_dataset("/home/mbaldacchino/data/vertical_velocity.grib")
vv = vv.rename_vars({"w": "w500"})

env_native = env_native.merge(
    vv.swap_dims({"time": "valid_time"})
    .sel(valid_time=slice("1980-01-01", "2025-12-01"))
    .drop_vars(["number", "time", "step"])
)

del vv


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

# ============================================================
# 1. CONFIGURATION - IMPORTANT - Carry the  XX years before for window
# ============================================================

START_YEAR = 1980
END_YEAR = 2024

CLIM_WINDOW = 40

# The SST DataArray must cover at least 1940–2024
# if START_YEAR = 1980 and CLIM_WINDOW = 40.
#
# Expected:
# dims = (valid_time, latitude, longitude)
# longitude convention = 0–360
# one value per calendar month


# sst_full = env_native["sst"]
sst_full = xr.open_dataset("/home/mbaldacchino/data/sst_1940_2026.nc")["sst"]


# EIO: East Indian Ocean
EIO = {
    "lat_min": -10,
    "lat_max": 22.5,
    "lon_min": 75,
    "lon_max": 100,
}

# SWP: Southwest Pacific
SWP = {
    "lat_min": -40,
    "lat_max": -20,
    "lon_min": 160,
    "lon_max": 190,  # 170°W
}

# WWP: Western Pacific Warm Pool
WWP = {
    "lat_min": 0,
    "lat_max": 16,
    "lon_min": 125,
    "lon_max": 165,
}


# ============================================================
# 2. SPATIAL PROCESSING
# ============================================================


def spatial_mask(da, boxes):
    """Union of one or more geographic boxes."""

    mask = xr.zeros_like(
        da.isel(valid_time=0, drop=True),
        dtype=bool,
    )

    for box in boxes:
        m = (
            (da.latitude >= box["lat_min"])
            & (da.latitude <= box["lat_max"])
            & (da.longitude >= box["lon_min"])
            & (da.longitude <= box["lon_max"])
        )

        mask = mask | m

    return mask


def regional_area_mean(da, boxes):
    """
    Area-weighted monthly mean over the union of boxes.

    Crops the DataArray before applying the spatial mask
    to avoid unnecessary operations on the global grid.
    """

    lat_min = min(b["lat_min"] for b in boxes)
    lat_max = max(b["lat_max"] for b in boxes)

    lon_min = min(b["lon_min"] for b in boxes)
    lon_max = max(b["lon_max"] for b in boxes)

    # Handle either ascending or descending latitude
    descending = da.latitude.values[0] > da.latitude.values[-1]

    lat_slice = slice(lat_max, lat_min) if descending else slice(lat_min, lat_max)

    sub = da.sel(
        latitude=lat_slice,
        longitude=slice(lon_min, lon_max),
    )

    if sub.latitude.size == 0 or sub.longitude.size == 0:
        raise ValueError(f"Empty spatial selection: {boxes}")

    mask = spatial_mask(sub, boxes)

    sub = sub.where(mask)

    weights = np.cos(np.deg2rad(sub.latitude))

    return sub.weighted(weights).mean(
        ["latitude", "longitude"],
        skipna=True,
    )


# ============================================================
# 3. ROLLING CLIMATOLOGY
# ============================================================


def rolling_monthly_ssta(
    monthly_sst: pd.Series,
    window: int = 40,
) -> pd.DataFrame:
    """
    Compute monthly SST anomalies using a trailing
    climatology of `window` complete calendar years.

    For target year Y and calendar month M:

        C(Y,M) = mean[SST(Y-40,M), ..., SST(Y-1,M)]

        SSTA(Y,M) = SST(Y,M) - C(Y,M)

    The target year is excluded.

    Missing historical months invalidate the corresponding
    reference window rather than silently shortening it.
    """

    s = monthly_sst.copy().sort_index()

    # Normalize all timestamps to the first day of their month
    s.index = pd.DatetimeIndex(s.index).to_period("M").to_timestamp()

    if s.index.has_duplicates:
        raise ValueError("Multiple SST observations for the same calendar month.")

    # Ensure missing calendar months are explicitly represented
    s = s.asfreq("MS")

    climatology = s.groupby(s.index.month).transform(
        lambda x: (
            x.shift(1)
            .rolling(
                window=window,
                min_periods=window,
            )
            .mean()
        )
    )

    return pd.DataFrame(
        {
            "sst": s,
            "climatology": climatology,
            "ssta": s - climatology,
        }
    )


# ============================================================
# 4. SEASONAL AGGREGATION
# ============================================================


def seasonal_index(
    monthly_series: pd.Series,
    months: list[int],
) -> pd.Series:
    """
    Calculate yearly means over the selected calendar months.

    A yearly value is only returned if every requested
    month is available and non-missing.

    Monthly values receive equal weights.
    """

    selected = monthly_series.loc[monthly_series.index.month.isin(months)]

    grouped = selected.groupby(selected.index.year)

    seasonal = grouped.mean()

    # Require a complete period
    seasonal = seasonal.where(grouped.count() == len(months))

    seasonal.index.name = "year"

    return seasonal


# ============================================================
# 5. EXTRACT REGIONAL SST TIME SERIES
# ============================================================

# Spatial aggregation is performed only once for each region.

monthly_sst = pd.DataFrame(
    {
        "eio": regional_area_mean(sst_full, [EIO]).to_series(),
        "swp": regional_area_mean(sst_full, [SWP]).to_series(),
        "wwp": regional_area_mean(sst_full, [WWP]).to_series(),
    }
).sort_index()


# ============================================================
# 6. COMPUTE MONTHLY ANOMALIES
# ============================================================

rolling_results = {
    name: rolling_monthly_ssta(
        monthly_sst[name],
        window=CLIM_WINDOW,
    )
    for name in ["eio", "swp", "wwp"]
}

monthly_ssta = pd.DataFrame(
    {name: result["ssta"] for name, result in rolling_results.items()}
)

# SST gradient anomaly:
# Southwest Pacific - Western Pacific Warm Pool

monthly_ssta["sstg"] = monthly_ssta["swp"] - monthly_ssta["wwp"]


# ============================================================
# 7. COMPUTE THE TWO WNP INDICES
# ============================================================

# Original Zhan et al. definitions:
#
# EIO  = June–August mean EIO SSTA
# SSTG = March–May mean (SWP SSTA - WWP SSTA)

eio_jja = seasonal_index(
    monthly_ssta["eio"],
    months=[6, 7, 8],
)

sstg_mam = seasonal_index(
    monthly_ssta["sstg"],
    months=[3, 4, 5],
)


# ============================================================
# 8. CONSTRUCT YEARLY DATAFRAME
# ============================================================

years = np.arange(
    START_YEAR,
    END_YEAR + 1,
)

wnp_predictors = pd.DataFrame({"year": years})

wnp_predictors["eio"] = wnp_predictors["year"].map(eio_jja)

wnp_predictors["sstg"] = wnp_predictors["year"].map(sstg_mam)

# New experiment: contemporaneous JJA SST gradient

sstg_jja = seasonal_index(
    monthly_ssta["sstg"],
    months=[6, 7, 8],
)

wnp_predictors["sstg_jja"] = wnp_predictors["year"].map(sstg_jja)


# ============================================================
# 9. CHECK HISTORICAL COVERAGE
# ============================================================

missing = wnp_predictors.loc[
    wnp_predictors[["eio", "sstg"]].isna().any(axis=1),
    "year",
].tolist()

if missing:
    raise ValueError(
        "Incomplete EIO/SSTG indices for years "
        f"{missing}.\n"
        f"A strict {CLIM_WINDOW}-year trailing climatology "
        f"requires monthly SST history beginning no later "
        f"than {START_YEAR - CLIM_WINDOW}. "
        "Check historical data coverage and missing months."
    )

print(wnp_predictors.head())
print(wnp_predictors.tail())
env_native = env_native.drop_vars(["sst"])


In [ ]:
eio_raw = seasonal_index(
    monthly_sst["eio"],
    [6, 7, 8],
)

sstg_raw = seasonal_index(
    monthly_sst["swp"] - monthly_sst["wwp"],
    [3, 4, 5],
)


eio_monthly = (
    regional_area_mean(
        sst_full,
        [EIO],
    )
    .to_series()
    .sort_index()
)

# Example using a monthly climatology estimated on training data
train_eio = eio_monthly  # [eio_monthly.index.year <= CFG.last_train_year]

monthly_clim = train_eio.groupby(train_eio.index.month).mean()

eio_anom = pd.Series(
    [value - monthly_clim.loc[t.month] for t, value in eio_monthly.items()],
    index=eio_monthly.index,
    name="eio_anom",
)

# Causal 3-month state: current + previous two months
eio_dynamic = eio_anom.rolling(3, min_periods=3).mean()


eio_dynamic_df = eio_dynamic.rename("eio_dynamic").reset_index()

eio_dynamic_df["year"] = eio_dynamic_df["valid_time"].dt.year

eio_dynamic_df["month"] = eio_dynamic_df["valid_time"].dt.month

eio_dynamic_df = eio_dynamic_df[["year", "month", "eio_dynamic"]]


In [ ]:
train_eta = prepare_clipped_eta(
    train,
    4.5e-5,
)


In [ ]:
region_grid = (
    train_eta[["latitude", "longitude", "region"]]
    .drop_duplicates(["latitude", "longitude"])
    .pivot(
        index="latitude",
        columns="longitude",
        values="region",
    )
)

import numpy as np
from scipy.ndimage import gaussian_filter

regions = sorted(train_eta["region"].unique())

hard_masks = {}

for region in regions:
    hard_masks[region] = (region_grid.values == region).astype(float)


SIGMA_REGION = 3.0

soft_masks = {}
for region in regions:
    soft_masks[region] = gaussian_filter(
        hard_masks[region],
        sigma=3.0,
        mode="nearest",
    )

total = np.zeros_like(next(iter(soft_masks.values())))

for field in soft_masks.values():
    total += field

total = np.where(
    total > 0,
    total,
    np.nan,
)

for region in regions:
    soft_masks[region] = soft_masks[region] / total


weights_df = []

lat_values = region_grid.index.to_numpy()
lon_values = region_grid.columns.to_numpy()

for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        row = {
            "latitude": lat,
            "longitude": lon,
        }

        for region in regions:
            row[f"w_{region}"] = soft_masks[region][i, j]

        weights_df.append(row)

weights_df = pd.DataFrame(weights_df)

train_soft = train_eta.merge(
    weights_df,
    on=["latitude", "longitude"],
    how="left",
)

test_soft = test_eta.merge(
    weights_df,
    on=["latitude", "longitude"],
    how="left",
)


REFERENCE_REGION = "scs"

varying_regions = [r for r in regions if r != REFERENCE_REGION]
region_terms = " + ".join(f"w_{r}" for r in varying_regions)


In [ ]:
train_soft = train_soft.merge(
    eio_dynamic_df,
    on=["year", "month"],
    how="left",
)

mu_eio = train_soft["eio_dynamic"].mean()
sd_eio = train_soft["eio_dynamic"].std()

train_soft["z_eio_dynamic"] = (train_soft["eio_dynamic"] - mu_eio) / sd_eio


In [ ]:
FORMULA_TREND_QUA = (
    "genesis_count ~ "  # C(month)"
    " + z_pi"
    " + z_rh600"
    " + z_vws"
    " + z_eta_clip"
    " + z_w500"
    f" + {region_terms}"
    " + z_eio_dynamic"
)

m_trend_eio_dyn = smf.glm(
    formula=FORMULA_TREND_QUA,
    data=train_soft,
    family=sm.families.Poisson(),
    offset=train_soft["log_exposure"],
).fit()


## C - EP

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


path_files = Path(
    "/home/mbaldacchino/data/SIENA-IH-STORM_1980_2025_VF/SIENA-IH-STORM/env_yearly"
)

# Files we actually need for the first genesis model.
NAME_MAP = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "VORT850": "vort850",  # relative vorticity [s-1]
    "SST": "sst",
}

NAME_MODEL = {
    "VMAX_PI": "pi",  # thermodynamic PI maximum wind [m/s]
    "VWS": "vws",  # 850-200 hPa shear [m/s]
    "RH600": "rh600",  # relative humidity
    "ETA850": "eta850",  # absolute vorticity capped [s-1]
    "W500": "w500",  # vertical velocity at 500hPa
}

START_YEAR = 1980
END_YEAR = 2025


def parse_env_filename(path):
    """
    Parse:
        VWS_2000_6.npy
        VMAX_PI_2000_6.npy
        VORT850_2018_10.npy

    Returns:
        variable, year, month
    """
    name, year, month = path.stem.rsplit("_", 2)

    return name, int(year), int(month)


In [ ]:
def load_environment_dataset(
    path_files,
    name_map,
    start_year=1980,
    end_year=2024,
):
    path_files = Path(path_files)

    # Your arrays are the global 0.25° fields used elsewhere in the repo:
    # 721 latitudes: 90 -> -90
    # 1440 longitudes: 0 -> 359.75
    lat = np.linspace(90.0, -90.0, 721)
    lon = np.arange(1440) * 0.25

    variables = {}

    for file_name, xr_name in name_map.items():
        records = []

        for file in path_files.glob(f"{file_name}_*.npy"):
            name, year, month = parse_env_filename(file)

            if name != file_name:
                continue

            if not (start_year <= year <= end_year):
                continue

            records.append((year, month, file))

        records.sort(key=lambda x: (x[0], x[1]))

        if not records:
            raise FileNotFoundError(f"No files found for {file_name}")

        arrays = []
        times = []

        for year, month, file in records:
            arr = np.load(file)

            if arr.shape != (721, 1440):
                raise ValueError(f"{file.name}: expected (721, 1440), got {arr.shape}")

            arrays.append(arr)

            times.append(pd.Timestamp(year=year, month=month, day=1))

        variables[xr_name] = xr.DataArray(
            np.stack(arrays, axis=0),
            dims=("valid_time", "latitude", "longitude"),
            coords={
                "valid_time": pd.DatetimeIndex(times),
                "latitude": lat,
                "longitude": lon,
            },
            name=xr_name,
        )

    return xr.Dataset(variables)


env_native = load_environment_dataset(
    path_files=path_files,
    name_map=NAME_MAP,
    start_year=START_YEAR,
    end_year=END_YEAR,
)

OMEGA = 7.2921e-5

f = 2 * OMEGA * np.sin(np.deg2rad(env_native.latitude))

eta = env_native["vort850"] + f

env_native["eta850"] = xr.where(
    eta > 0,
    eta,
    0.0,
).clip(max=5e-5)

# TEMP FIX ADD W500
vv = xr.open_dataset("/home/mbaldacchino/data/vertical_velocity.grib")
vv = vv.rename_vars({"w": "w500"})

env_native = env_native.merge(
    vv.swap_dims({"time": "valid_time"})
    .sel(valid_time=slice("1980-01-01", "2025-12-01"))
    .drop_vars(["number", "time", "step"])
)

del vv


In [ ]:
# COMPUTE rSST

JJASON = [6, 7, 8, 9, 10, 11]
ASO = [8, 9, 10]


def area_weighted_mean(da):
    weights = np.cos(np.deg2rad(da.latitude))

    return da.weighted(weights).mean(["latitude", "longitude"])


def area_weighted_mean_quantile(
    da,
    quantile=0.7,
):
    """
    Area-weighted mean of values above an area-weighted quantile.

    For SSTW30:
        quantile=0.7
    returns approximately the area-weighted mean SST of the
    warmest 30% of the domain.

    Assumes:
    - regular lat/lon grid
    - land points are NaN
    """

    weights = np.cos(np.deg2rad(da.latitude))

    # Area-weighted quantile at each time step
    threshold = da.weighted(weights).quantile(
        quantile,
        dim=["latitude", "longitude"],
        skipna=True,
    )

    # Keep only cells warmer than the weighted threshold
    warm = da.where(da >= threshold)

    # Area-weighted mean of those warmest cells
    return warm.weighted(weights).mean(
        ["latitude", "longitude"],
        skipna=True,
    )


MDR_SST = area_weighted_mean(
    env_native.sst.sel(
        latitude=slice(
            MDR_REGION["lat_max"],
            MDR_REGION["lat_min"],
        ),
        longitude=slice(
            MDR_REGION["lon_min"],
            MDR_REGION["lon_max"],
        ),
    )
)

TROPICS_SST = area_weighted_mean(
    env_native.sst.sel(
        latitude=slice(
            TROPICS_REGION["lat_max"],
            TROPICS_REGION["lat_min"],
        ),
        longitude=slice(
            TROPICS_REGION["lon_min"],
            TROPICS_REGION["lon_max"],
        ),
    )
)

SST_W30 = area_weighted_mean_quantile(
    env_native.sst.sel(
        latitude=slice(
            TROPICS_REGION["lat_max"],
            TROPICS_REGION["lat_min"],
        ),
        longitude=slice(
            TROPICS_REGION["lon_min"],
            TROPICS_REGION["lon_max"],
        ),
    ),
    quantile=0.70,
)

ds_rsst = xr.Dataset(
    {
        "mdr_sst": MDR_SST,
        "tropics_sst": TROPICS_SST,
        "rsst": MDR_SST - TROPICS_SST,
        "rsst_bis": MDR_SST - SST_W30,
    }
)

ds_jjason = ds_rsst.sel(valid_time=ds_rsst.valid_time.dt.month.isin(JJASON))
weights = ds_jjason.valid_time.dt.days_in_month
yearly = (ds_jjason * weights).groupby("valid_time.year").sum(
    "valid_time"
) / weights.groupby("valid_time.year").sum("valid_time")


ds_aso = ds_rsst.sel(valid_time=ds_rsst.valid_time.dt.month.isin(ASO))
weights = ds_aso.valid_time.dt.days_in_month
yearly_aso = (ds_aso * weights).groupby("valid_time.year").sum(
    "valid_time"
) / weights.groupby("valid_time.year").sum("valid_time")

yearly_df = yearly.to_dataframe().reset_index()
yearly_aso = yearly_aso.to_dataframe().reset_index()


env_native = env_native.drop_vars("sst")


In [ ]:
region_grid = (
    train_eta[["latitude", "longitude", "region"]]
    .drop_duplicates(["latitude", "longitude"])
    .pivot(
        index="latitude",
        columns="longitude",
        values="region",
    )
)

import numpy as np
from scipy.ndimage import gaussian_filter

regions = sorted(train_eta["region"].unique())

hard_masks = {}

for region in regions:
    hard_masks[region] = (region_grid.values == region).astype(float)

soft_masks = {}

for region in regions:
    soft_masks[region] = gaussian_filter(
        hard_masks[region],
        sigma=3.0,
        mode="nearest",
    )

total = np.zeros_like(next(iter(soft_masks.values())))

for field in soft_masks.values():
    total += field

total = np.where(
    total > 0,
    total,
    np.nan,
)

for region in regions:
    soft_masks[region] = soft_masks[region] / total


weights_df = []

lat_values = region_grid.index.to_numpy()
lon_values = region_grid.columns.to_numpy()

for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        row = {
            "latitude": lat,
            "longitude": lon,
        }

        for region in regions:
            row[f"w_{region}"] = soft_masks[region][i, j]

        weights_df.append(row)

weights_df = pd.DataFrame(weights_df)


In [ ]:


train_eta["rsst_bis"] = train_eta["year"].map(yearly_df.set_index("year")["rsst_bis"])
train_eta["rsst_bis"] = (
    train_eta["rsst_bis"] - np.mean(train_eta["rsst_bis"])
) / np.std(train_eta["rsst_bis"])


train_soft = train_eta.merge(
    weights_df,
    on=["latitude", "longitude"],
    how="left",
)


In [ ]:
REFERENCE_REGION = "subtropical"
varying_regions = [r for r in regions if r != REFERENCE_REGION]
region_terms = " + ".join(f"w_{r}" for r in varying_regions)

interaction_terms = " + ".join(f"w_{r}:C(month)" for r in varying_regions)


FORMULA_TREND = (
    f"genesis_count ~ + z_pi + z_rh600 + z_vws + z_eta_clip + rsst_bis + {region_terms}"
)

m_trend = smf.glm(
    formula=FORMULA_TREND,
    data=train_soft,
    family=sm.families.Poisson(),
    offset=train_soft["log_exposure"],
).fit()


## D - NI

In [ ]:
def ni_season(month):
    if month in [3, 4, 5, 6]:
        return "pre_monsoon"
    elif month in [7, 8, 9]:
        return "monsoon"
    else:
        return "post_monsoon"


train_soft["ni_season"] = train_soft["month"].map(ni_season)
train_soft = prepare_clipped_eta(
    train_soft,
    4.5e-5,
)

In [ ]:
FORMULA_NI_SEASON = (
    "genesis_count ~ C(ni_season) + C(region) + z_eta_clip +  z_vws + z_w500 "
)

m_ni_season = smf.glm(
    formula=FORMULA_NI_SEASON,
    data=train_soft,
    family=sm.families.Poisson(),
    offset=train_soft["log_exposure"],
).fit()

print("4.5e-5 threshold")
print(m_ni_season.summary())


## E - SI

In [ ]:
train_eta = prepare_clipped_eta(
    train,
    5e-5,
)


In [ ]:
REGIONS = [
    "southwest_indian_ocean",
    "central_south_indian_ocean",
    "northwest_australia",
]


In [ ]:
region_grid = (
    train_eta[["latitude", "longitude", "region"]]
    .drop_duplicates(["latitude", "longitude"])
    .pivot(
        index="latitude",
        columns="longitude",
        values="region",
    )
)

import numpy as np
from scipy.ndimage import gaussian_filter

regions = sorted(train_eta["region"].unique())

hard_masks = {}

for region in regions:
    hard_masks[region] = (region_grid.values == region).astype(float)


SIGMA_REGION = 3.0


In [ ]:
soft_masks = {}

for region in regions:
    soft_masks[region] = gaussian_filter(
        hard_masks[region],
        sigma=3.0,
        mode="nearest",
    )

total = np.zeros_like(next(iter(soft_masks.values())))

for field in soft_masks.values():
    total += field

total = np.where(
    total > 0,
    total,
    np.nan,
)

for region in regions:
    soft_masks[region] = soft_masks[region] / total


weights_df = []

lat_values = region_grid.index.to_numpy()
lon_values = region_grid.columns.to_numpy()

for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        row = {
            "latitude": lat,
            "longitude": lon,
        }

        for region in regions:
            row[f"w_{region}"] = soft_masks[region][i, j]

        weights_df.append(row)

weights_df = pd.DataFrame(weights_df)

train_soft = train_eta.merge(
    weights_df,
    on=["latitude", "longitude"],
    how="left",
)


In [ ]:
REFERENCE_REGION = "central_south_indian_ocean"

varying_regions = [r for r in regions if r != REFERENCE_REGION]
region_terms = " + ".join(f"w_{r}" for r in varying_regions)


In [ ]:
FORMULA_SOFT_REGION_MONTH = (
    "genesis_count ~ "  # C(month)"
    " + z_pi"
    " + z_rh600"
    " + z_vws"
    " + z_eta_clip"
    " + z_w500"
    f" + {region_terms}"
)
m_soft_region_month = smf.glm(
    formula=FORMULA_SOFT_REGION_MONTH,
    data=train_soft,
    family=sm.families.Poisson(),
    offset=train_soft["log_exposure"],
).fit()


## F - SP

In [ ]:
train_eta, test_eta = prepare_clipped_eta(
    train,
    test,
    4e-5,
)


In [ ]:
train_eta["region"] = [
    assign_genesis_region(lat, lon)
    for lat, lon in zip(
        train_eta["latitude"],
        train_eta["longitude"],
    )
]


In [ ]:
region_grid = (
    train_eta[["latitude", "longitude", "region"]]
    .drop_duplicates(["latitude", "longitude"])
    .pivot(
        index="latitude",
        columns="longitude",
        values="region",
    )
)

import numpy as np
from scipy.ndimage import gaussian_filter

regions = sorted(train_eta["region"].unique())

hard_masks = {}

for region in regions:
    hard_masks[region] = (region_grid.values == region).astype(float)


SIGMA_REGION = 3.0


In [ ]:
soft_masks = {}

for region in regions:
    soft_masks[region] = gaussian_filter(
        hard_masks[region],
        sigma=3.0,
        mode="nearest",
    )

total = np.zeros_like(next(iter(soft_masks.values())))

for field in soft_masks.values():
    total += field

total = np.where(
    total > 0,
    total,
    np.nan,
)

for region in regions:
    soft_masks[region] = soft_masks[region] / total


weights_df = []

lat_values = region_grid.index.to_numpy()
lon_values = region_grid.columns.to_numpy()

for i, lat in enumerate(lat_values):
    for j, lon in enumerate(lon_values):
        row = {
            "latitude": lat,
            "longitude": lon,
        }

        for region in regions:
            row[f"w_{region}"] = soft_masks[region][i, j]

        weights_df.append(row)

weights_df = pd.DataFrame(weights_df)

train_soft = train_eta.merge(
    weights_df,
    on=["latitude", "longitude"],
    how="left",
)


In [ ]:
roni = pd.read_csv("/home/mbaldacchino/data/roni.csv")


for df in [train_soft]:
    df["season_year"] = np.where(
        df["month"] >= 9,
        df["year"],
        df["year"] - 1,
    )


In [ ]:
def add_seasonal_roni(
    train,
    roni,
    seas,
    output_name,
):
    # RONI value indexed by cyclone-season starting year
    mapping = roni.loc[roni["SEAS"] == seas].set_index("YR")["ANOM"]

    train[output_name] = train["season_year"].map(mapping)

    # Standardize using TRAINING climate years only
    train_year_values = (
        train[["season_year", output_name]]
        .drop_duplicates("season_year")[output_name]
        .dropna()
    )

    mean_ = train_year_values.mean()
    std_ = train_year_values.std()

    train[f"{output_name}_standardized"] = (train[output_name] - mean_) / std_

    return mean_, std_


In [ ]:
roni_ond_mean, roni_ond_std = add_seasonal_roni(
    train_soft,
    roni,
    seas="OND",
    output_name="roni_ond",
)


In [ ]:
FORMULA_REGION_RONI_OND = (
    "genesis_count ~ "
    " + z_pi:w_central_south_pacific"
    " + z_pi:w_eastern_south_pacific"
    " + z_vws"
    " + z_eta_clip"
    " + z_w500"
    " + w_eastern_south_pacific"
    " + roni_ond_standardized:w_eastern_south_pacific"
)

m_region_roni_ond = smf.glm(
    formula=FORMULA_REGION_RONI_OND,
    data=train_soft,
    family=sm.families.Poisson(),
    offset=train_soft["log_exposure"],
).fit()
